# Lab 3.3 · Solving Non-invertible Linear Systems

Solve Ax = b without an inverse: identify pivot and free variables, test whether b is reachable, and describe the entire solution set.

Wooseok Ha, MAS110 Lecture 3: Vector Spaces (Fall 2026), lec03.pdf. Main example: printed slides 17–31 / PDF pages 21–35. Solution-set theorem: printed slides 15–16 / PDF pages 18–19.

Maintained in [MAS110 Practice Sessions](https://github.com/statchan1106/mas110-practice-sessions). Course companion to [Foundations of LADS](https://github.com/kyunghyuncho/Foundations_of_LADS) by Wanmo Kang and Kyunghyun Cho. This notebook follows the supplied lecture example; the upstream repository is a reference.

Run the cells from top to bottom. Only NumPy is required (available in Colab). The row operations below are chosen for this particular matrix, not a general-purpose solver.

“Non-invertible” includes the rectangular system studied here: A has three rows and four columns, so there is no ordinary two-sided inverse. A square non-invertible matrix is also called singular. Elimination answers two separate questions: does a solution exist for this b, and, if it does, how many solutions are there?

## Notation

- **A ∈ ℝᵐˣⁿ; here m = 3, n = 4**: m counts equations (rows); n counts unknowns (columns). A maps inputs in ℝⁿ to outputs in ℝᵐ. ℝ means the real numbers; ∈ means “belongs to.”

- **x = (u, v, w, y)ᵀ; b = (b₁, b₂, b₃)ᵀ**: x is the unknown input; b is the given target. Superscript T transposes a written row into a column. The scalar y is the fourth coordinate, not an entire vector.

- **aⱼ; U; R**: aⱼ is column j of the original A. U is a row echelon form; R is its reduced row echelon form. Lowercase u is a coordinate of x; uppercase U is a matrix.

- **E = L⁻¹; c = Eb; [A | b] → [U | c]**: E collects the downward row operations and satisfies EA = U. E is invertible even though A has no inverse. The vertical bar separates coefficients from the right-hand side; b must undergo the same operations.

- **r = rank(A) = 2; n − r = 2**: r counts pivots and equals dim Col(A). n − r counts free variables and equals dim Null(A). Dimension and number of coordinates differ: a two-dimensional plane can lie inside ℝ⁴.

- **xₚ; n₁, n₂; α, β ∈ ℝ**: xₚ is one particular solution. n₁ and n₂ are null-space basis vectors (called x₁ and x₂ on printed slide 27); this page uses n₁ and n₂ to avoid confusion with coordinates. α and β are free scalars, equal to v and y.

- **span{n₁, n₂}; xₚ + Null(A)**: span means every linear combination αn₁ + βn₂. Adding xₚ to a set means adding it to every vector in that set. ⇔ means “if and only if”: both directions hold.

- **NumPy: (3, 4), (4,), @, [1, 2]**: A.shape is (3, 4). A 1D vector array has shape (4,), not (4, 1); transposing it does not create a column array. @ is matrix multiplication; * is elementwise multiplication or scalar scaling. A[1, 2] selects lecture row 2, column 3 because Python counts from 0.

## Key ideas

### Column space: existence

Col(A) contains every possible output Ax. A solution exists precisely when the target b belongs to that set.

Col(A) = {Ax : x ∈ ℝ⁴} ⊆ ℝ³

Keep in mind: Here the reachable outputs form the plane 5b₁ − 2b₂ + b₃ = 0.

### Null space: freedom in the input

Null(A) contains the inputs sent to zero. Adding one to a solution leaves its output unchanged.

Null(A) = {z ∈ ℝ⁴ : Az = 0}

Keep in mind: The zero on the right belongs to ℝ³; the null-space vectors belong to ℝ⁴.

### Pivot and free variables

In echelon form, the first nonzero entry of each nonzero row is its pivot. Solve for the pivot variables; choose the others freely in a consistent system.

pivot: u, w; free: v = α, y = β

Keep in mind: The second pivot is in column 3. A pivot need not lie on the diagonal or equal 1.

### The complete solution

Once a particular solution xₚ exists, add every linear combination of the null-space basis vectors n₁ and n₂.

x = xₚ + αn₁ + βn₂, α, β ∈ ℝ

Keep in mind: There are n − r = 4 − 2 = 2 free parameters. One null direction would miss solutions.

## 1. Keep A and a working copy

A represents the original map. U starts as a copy and will become its row echelon form. Python counts from 0; the lecture counts from 1.

**Predict first:** Which array should check the original equation Ax = b?

- Line 1, `import numpy as np`: Loads NumPy. Shape: module import. Operation: np is the short name for NumPy.

- Line 2, `A = np.array([[1., 3., 3., 2.], [2., 6., 9., 7.], [-1., -3., 3., 4.]])`: Stores the lecture matrix row by row. Shape: 3 rows × 4 entries → A: (3, 4). Operation: A @ x maps a length-4 input to a length-3 output. Decimal points allow later division in place.

- Line 3, `U = A.copy()`: Makes a separate working array. Shape: (3, 4) → U: (3, 4). Operation: Changing U will not overwrite A; keep A for its original columns and a final Ax check.

In [ ]:
import numpy as np
A = np.array([[1., 3., 3., 2.], [2., 6., 9., 7.], [-1., -3., 3., 4.]])
U = A.copy()

In [ ]:
print("A =", A, "U =", U, sep="\n")

**Read the result:** Both arrays are initially 3 × 4. Row operations will change only U.



U[1] is row 2; U[1, 2] is the entry in row 2, column 3.

## 2. Eliminate downward and skip a column without a pivot

Column 2 has no pivot candidate below row 1, so move to column 3. Its pivot is 3. The final zero row means only two equations are independent.

**Predict first:** Why is the second pivot in column 3 instead of column 2?

- Line 1, `U[1] -= 2 * U[0]`: Clears the 2 below the first pivot. Shape: (4,) − scalar × (4,) → (4,). Operation: row₂ ← row₂ − 2 row₁: [2,6,9,7] − 2[1,3,3,2] = [0,0,3,3].

- Line 2, `U[2] += U[0]`: Clears the −1 below the first pivot. Shape: (4,) + (4,) → (4,). Operation: row₃ ← row₃ + row₁: [−1,−3,3,4] + [1,3,3,2] = [0,0,6,6].

- Line 3, `U[2] -= 2 * U[1]`: Uses the updated row 2 and its pivot in column 3. Shape: (4,) − scalar × (4,) → (4,). Operation: row₃ ← row₃ − 2 row₂: [0,0,6,6] − 2[0,0,3,3] = [0,0,0,0].

In [ ]:
U[1] -= 2 * U[0]
U[2] += U[0]
U[2] -= 2 * U[1]

In [ ]:
print("U =", U, sep="\n")

**Read the result:** Pivots are at (1,1) and (2,3); the zero row is at the bottom. U is rectangular, so use the term row echelon form.

U = EA, where E = L⁻¹ is invertible

For Ax = b, the same row operations must act on b.

## 3. Read the homogeneous equations from R

Each pivot is 1 and the only nonzero entry in its column. Rx = 0 says u + 3v − y = 0 and w + y = 0. Solve for u and w in terms of v and y.

**Predict first:** If v and y are chosen, what determines u and w?

- Line 1, `R = U.copy()`: Preserves U and prepares a reduced form. Shape: (3, 4) → R: (3, 4). Operation: R initially contains U.

- Line 2, `R[1] /= 3`: Scales the second pivot to 1. Shape: (4,) ÷ scalar → (4,). Operation: [0,0,3,3] / 3 = [0,0,1,1].

- Line 3, `R[0] -= 3 * R[1]`: Clears the entry above the second pivot. Shape: (4,) − scalar × (4,) → (4,). Operation: [1,3,3,2] − 3[0,0,1,1] = [1,3,0,−1].

In [ ]:
R = U.copy()
R[1] /= 3
R[0] -= 3 * R[1]

In [ ]:
print("R =", R, sep="\n")

**Read the result:** Set v = α and y = β, with any real α and β.

u = −3α + β, v = α, w = −β, y = β

Null(A) = Null(U) = Null(R). This does not mean Col(A) = Col(R).

## 4. Build the null-space basis

Every homogeneous solution is αn₁ + βn₂. These vectors are independent: the v and y coordinates of their combination are exactly α and β.

**Predict first:** If αn₁ + βn₂ = 0, what must α and β be?

- Line 1, `n1 = np.array([-3., 1., 0., 0.])`: Sets free coordinates (v,y) = (1,0). Shape: 4 values → n1: (4,). Operation: u = −3, w = 0: n₁ = (−3,1,0,0)ᵀ.

- Line 2, `n2 = np.array([1., 0., -1., 1.])`: Sets free coordinates (v,y) = (0,1). Shape: 4 values → n2: (4,). Operation: u = 1, w = −1: n₂ = (1,0,−1,1)ᵀ.

- Line 3, `N = np.column_stack((n1, n2))`: Stores the basis vectors as columns. Shape: two (4,) arrays → N: (4, 2). Operation: N @ [α,β] computes αn₁ + βn₂.

- Line 4, `null_check = A @ N`: Checks both vectors against the original A. Shape: (3, 4) @ (4, 2) → (3, 2). Operation: Each output column is zero: An₁ = An₂ = 0.

In [ ]:
n1 = np.array([-3., 1., 0., 0.])
n2 = np.array([1., 0., -1., 1.])
N = np.column_stack((n1, n2))
null_check = A @ N

In [ ]:
print("N =", N, "A @ N =", null_check, sep="\n")

**Read the result:** N has four rows because its columns are inputs; AN has three rows because these map into the output space.

Null(A) = span{n₁,n₂}; dim Null(A) = 2

Build the null-space basis from Ax = 0, even when the target problem has b ≠ 0.

## 5. Transform b before testing consistency

Ax = b is equivalent to Ux = c with c = Eb. The last row requires 0 = c₃, so a solution exists exactly when 5b₁ − 2b₂ + b₃ = 0.

**Predict first:** Which transformed right-hand side makes the zero coefficient row contradictory?

- Line 1, `E = np.array([[1., 0., 0.], [-2., 1., 0.], [5., -2., 1.]])`: Collects the downward row operations. Shape: E: (3, 3). Operation: E = L⁻¹ and E @ A = U. E acts on three equations, not four variables.

- Line 2, `b_good = np.array([1., 5., 5.])`: Chooses a reachable target. Shape: b_good: (3,). Operation: 5b₁ − 2b₂ + b₃ = 5 − 10 + 5 = 0.

- Line 3, `b_bad = np.array([1., 5., 6.])`: Changes the third output coordinate. Shape: b_bad: (3,). Operation: 5b₁ − 2b₂ + b₃ = 5 − 10 + 6 = 1.

- Line 4, `c_good, c_bad = E @ b_good, E @ b_bad`: Applies the same elimination to both targets. Shape: two (3, 3) @ (3,) products → two (3,) arrays. Operation: c_good = [1,3,0]; c_bad = [1,3,1].

In [ ]:
E = np.array([[1., 0., 0.], [-2., 1., 0.], [5., -2., 1.]])
b_good = np.array([1., 5., 5.])
b_bad = np.array([1., 5., 6.])
c_good, c_bad = E @ b_good, E @ b_bad

In [ ]:
print("E @ A =", E @ A, "c_good =", c_good, "c_bad =", c_bad, sep="\n")

**Read the result:** These matrices are [U | c], not [A | b]. The reachable target gives a redundant last equation.

c = (b₁, b₂ − 2b₁, 5b₁ − 2b₂ + b₃)ᵀ

A zero row in U alone does not mean no solution. The corresponding entry of c decides.

## 6. Connect consistency to column space and rank

Take pivot column indices 1 and 3, then select those columns from the original A. They form a basis of Col(A), the plane 5b₁ − 2b₂ + b₃ = 0.

**Predict first:** Why do we take the basis columns from A rather than R?

- Line 1, `rank_A = np.linalg.matrix_rank(A)`: Computes the numerical rank of A. Shape: (3, 4) → scalar. Operation: rank_A = 2 agrees with the pivot count.

- Line 2, `rank_good = np.linalg.matrix_rank(np.column_stack((A, b_good)))`: Appends b_good as an extra column. Shape: (3, 5) → scalar. Operation: rank_good = 2: no new independent output direction.

- Line 3, `rank_bad = np.linalg.matrix_rank(np.column_stack((A, b_bad)))`: Appends b_bad instead. Shape: (3, 5) → scalar. Operation: rank_bad = 3: the new column lies outside Col(A).

In [ ]:
rank_A = np.linalg.matrix_rank(A)
rank_good = np.linalg.matrix_rank(np.column_stack((A, b_good)))
rank_bad = np.linalg.matrix_rank(np.column_stack((A, b_bad)))

In [ ]:
print("rank(A), rank([A|b_good]), rank([A|b_bad]) =", rank_A, rank_good, rank_bad)

**Read the result:** Appending a reachable b leaves the column span unchanged.

b ∈ Col(A) ⇔ rank([A | b]) = rank(A) ⇔ 5b₁ − 2b₂ + b₃ = 0

The theorem uses exact rank. NumPy estimates rank with a singular-value tolerance; near-dependent or noisy data need care with scale and tolerance.

## 7. Find a particular solution

Setting free variables to zero gives one convenient solution, not a uniquely preferred one. For any consistent b, substitution gives u = 3b₁ − b₂ and w = (b₂ − 2b₁)/3.

**Predict first:** Why must consistency be checked before constructing xₚ?

- Line 1, `b = b_good`: Continues with the consistent target. Shape: b: (3,). Operation: b = [1,5,5]. b_bad has no particular solution.

- Line 2, `w = c_good[1] / U[1, 2]`: Sets v = y = 0 and solves row 2. Shape: scalar ÷ scalar → scalar. Operation: 3w = 3, so w = 1. U[1, 2] is the second pivot.

- Line 3, `u = (c_good[0] - U[0, 2] * w) / U[0, 0]`: Back-substitutes into row 1. Shape: scalar arithmetic → scalar. Operation: u + 3w = 1, so u = −2.

- Line 4, `x_particular = np.array([u, 0., w, 0.])`: Restores the coordinate order (u,v,w,y). Shape: 4 values → x_particular: (4,). Operation: xₚ = (−2,0,1,0)ᵀ.

In [ ]:
b = b_good
w = c_good[1] / U[1, 2]
u = (c_good[0] - U[0, 2] * w) / U[0, 0]
x_particular = np.array([u, 0., w, 0.])

In [ ]:
print("x_particular =", x_particular, "A @ x_particular =", A @ x_particular, sep="\n")

**Read the result:** Add null-space vectors next to obtain the remaining solutions.

xₚ = (3b₁ − b₂, 0, (b₂ − 2b₁)/3, 0)ᵀ

This formula assumes 5b₁ − 2b₂ + b₃ = 0. It does not solve an inconsistent system.

## 8. Describe every solution with two parameters

Any real α and β change only the null-space part, keeping Ax = b. Conversely, subtracting xₚ from any solution produces a null-space vector.

**Predict first:** If α or β changes, which input coordinates change and which output stays fixed?

- Line 1, `alpha, beta = 2., -1.`: Chooses the two free coordinates. Shape: two scalars. Operation: v = α = 2; y = β = −1.

- Line 2, `x = x_particular + alpha * n1 + beta * n2`: Adds a homogeneous solution to xₚ. Shape: (4,) + scalar × (4,) + scalar × (4,) → (4,). Operation: [−2,0,1,0] + 2[−3,1,0,0] − [1,0,−1,1] = [−9,2,2,−1].

- Line 3, `output = A @ x`: Checks the original equation. Shape: (3, 4) @ (4,) → (3,). Operation: A @ [−9,2,2,−1] = [1,5,5].

- Line 4, `verified = np.allclose(output, b, rtol=0, atol=1e-10)`: Checks an explicit absolute tolerance. Shape: two (3,) arrays → boolean. Operation: verified = True. The calculation checks this input; the proof below covers the full family.

In [ ]:
alpha, beta = 2., -1.
x = x_particular + alpha * n1 + beta * n2
output = A @ x
verified = np.allclose(output, b, rtol=0, atol=1e-10)

In [ ]:
print("x =", x, "A @ x =", output, "verified =", verified, sep="\n")

**Read the result:** The solution set is a plane inside ℝ⁴. This diagram shows parameter coordinates (α,β), not the four-dimensional input space.

x = xₚ + αn₁ + βn₂, α, β ∈ ℝ; Ax = (1,5,5)ᵀ

xₚ corresponds to (α,β) = (0,0). Every point of the entire parameter plane specifies a solution, not only the two marked points.

## Why this gives every solution

### Elimination changes equations without losing solutions

Each row operation can be undone, so their product E is invertible. Multiplying both sides of Ax = b by E gives Ux = c; multiplying by E⁻¹ recovers the original system. Here no row exchanges are needed, so E = L⁻¹ and A = LU. If rows are exchanged in another problem, their permutation must act on b too.

Further invertible row operations take U to R. Thus Ax = 0, Ux = 0, and Rx = 0 have the same solutions. For a nonzero right-hand side, reducing U further also requires reducing c; Rx = c would generally be a different system. Row operations preserve null space and rank but generally change column space: Col(EA) = E Col(A).

Ax = b ⇔ EAx = Eb ⇔ Ux = c

### Existence is a condition on the target b

The last row of U is zero, so the transformed system requires c₃ = 5b₁ − 2b₂ + b₃ = 0. This condition is also sufficient: when it holds, choose v and y freely, then use the two nonzero pivots to solve for w and u. For b_good = (1,5,5)ᵀ it holds; for b_bad = (1,5,6)ᵀ the last equation is 0 = 1.

This condition describes a plane through the origin in ℝ³: Col(A). Pivot columns 1 and 3 of the original A give a basis a₁ = (1,2,−1)ᵀ, a₃ = (3,9,3)ᵀ. Column 2 equals 3a₁; column 4 equals a₃ − a₁. The vector ℓ = (5,−2,1)ᵀ satisfies Aᵀℓ = 0, so the consistency condition is also ℓᵀb = 0.

Ax = b is consistent ⇔ b ∈ Col(A) ⇔ rank([A | b]) = rank(A)

### Free variables produce a basis, not just sample solutions

For Rx = 0, the pivot equations give u = −3v + y and w = −y. Substitute v = α and y = β: x = α(−3,1,0,0)ᵀ + β(1,0,−1,1)ᵀ. Every homogeneous solution is in this span because the formula comes from all its equations.

The two vectors are independent: if αn₁ + βn₂ = 0, the second coordinate forces α = 0 and the fourth forces β = 0. They form a basis of Null(A). Rank–nullity gives 2 + 2 = 4, the number of input coordinates, not 3, the number of equations.

Null(A) = {αn₁ + βn₂ : α, β ∈ ℝ}; rank(A) + dim Null(A) = n

### Prove the complete solution formula in both directions

Assume consistency and choose xₚ with Axₚ = b. For any z in Null(A), A(xₚ + z) = Axₚ + Az = b + 0 = b. Every vector in the proposed family is therefore a solution.

Conversely, take any solution x. Then A(x − xₚ) = b − b = 0, so x − xₚ belongs to Null(A) and equals αn₁ + βn₂. The family therefore misses no solutions. A different particular solution changes the starting point, but not the set.

{x : Ax = b} = xₚ + Null(A) = {xₚ + αn₁ + βn₂ : α, β ∈ ℝ}

### Separate existence, uniqueness, and geometry

For a consistent system, the solution is unique exactly when Null(A) = {0}, equivalently r = n. If r < n, a nonzero null-space direction produces infinitely many solutions over ℝ. Without consistency there is no solution: Null(A) = {0} alone does not guarantee existence for an arbitrary b.

Here r = 2 < 4, so every reachable b has infinitely many solutions forming an affine plane in ℝ⁴. For b ≠ 0 it misses the origin and is not a vector subspace. For b = 0 it is Null(A). Separately, the reachable targets form a plane in ℝ³. These planes live in different spaces.

inconsistent: none; consistent and r = n: one; consistent and r < n: infinitely many

### Apply the method to another m × n system

1. Row-reduce [A | b], keeping the variable order fixed and applying every operation to the whole augmented row. Identify the r coefficient pivots. 2. Check every all-zero coefficient row: a nonzero right-hand side proves inconsistency, so stop. Otherwise the system is consistent.

3. Set the n − r free variables to zero and solve the pivot equations for xₚ. 4. Return to Ax = 0; set one free variable to 1 and all other free variables to 0 in turn, solving for the pivot variables each time. This gives a basis n₁,…,nₙ₋ᵣ. 5. Write x = xₚ + t₁n₁ + ⋯ + tₙ₋ᵣnₙ₋ᵣ with arbitrary real parameters, and check using the original A and b.

The walkthrough follows a known pivot sequence for this matrix, not a general elimination routine. With floating-point or noisy data, choose scale-appropriate tolerances. np.linalg.matrix_rank estimates rank numerically; np.allclose checks approximate equality. Neither replaces the exact proof of the full solution set.



## Numerical checks

The proof above establishes the full family. These checks verify the displayed example and several choices of the free parameters. Approximate equality uses an explicit absolute tolerance.

In [ ]:
assert A.shape == (3, 4) and N.shape == (4, 2)
assert np.allclose(E @ A, U, rtol=0, atol=1e-10)
assert np.allclose(A @ N, np.zeros((3, 2)), rtol=0, atol=1e-10)
assert np.linalg.matrix_rank(N) == 2
assert (rank_A, rank_good, rank_bad) == (2, 2, 3)
assert np.allclose(c_good, [1, 3, 0], rtol=0, atol=1e-10)
assert np.allclose(c_bad, [1, 3, 1], rtol=0, atol=1e-10)
assert np.allclose(x_particular, [-2, 0, 1, 0], rtol=0, atol=1e-10)
assert np.allclose(x, [-9, 2, 2, -1], rtol=0, atol=1e-10)
ell = np.array([5., -2., 1.])
assert np.allclose(A.T @ ell, np.zeros(4), rtol=0, atol=1e-10)
for a, beta_value in [(0, 0), (0, 1), (2, -1), (-1.5, 3)]:
    candidate = x_particular + a * n1 + beta_value * n2
    assert np.allclose(A @ candidate, b, rtol=0, atol=1e-10)
print("All example checks passed.")

## Try another consistent target

Choose b₁ and b₂ first. Set b₃ = 2b₂ − 5b₁ to satisfy the consistency condition, then use the general particular-solution formula. Change the free parameters too.

In [ ]:
b1, b2 = 2., -1.
b_new = np.array([b1, b2, 2 * b2 - 5 * b1])
xp_new = np.array([3 * b1 - b2, 0., (b2 - 2 * b1) / 3, 0.])
alpha_new, beta_new = -0.5, 2.
x_new = xp_new + alpha_new * n1 + beta_new * n2
assert np.allclose(A @ x_new, b_new, rtol=0, atol=1e-10)
print("b_new =", b_new, "x_new =", x_new, "A @ x_new =", A @ x_new, sep="\n")

## Check your understanding

**Does a zero row of U automatically mean no solution?**

<details><summary>Show explanation</summary>

No. Check the matching entry of c = Eb. Zero gives the redundant equation 0 = 0; a nonzero entry gives a contradiction. Here check 5b₁ − 2b₂ + b₃.

</details>

**Can R’s pivot columns be used as a basis of Col(A)?**

<details><summary>Show explanation</summary>

Use their column indices, then take those columns of the original A. R’s pivot columns span the plane whose third coordinate is zero; Col(A) is the different plane 5b₁ − 2b₂ + b₃ = 0.

</details>

**For b = (1,5,5)ᵀ, what happens at α = 0 and β = 1?**

<details><summary>Show explanation</summary>

x = xₚ + n₂ = (−1,0,0,1)ᵀ. Its image is −a₁ + a₄ = (1,5,5)ᵀ. The input changes, but b does not.

</details>

**Why do we need two free parameters?**

<details><summary>Show explanation</summary>

Four unknowns minus two pivots leaves n − r = 2 free variables. Both independent null-space directions are needed: xₚ + n₂ cannot be obtained from xₚ + αn₁ alone.

</details>

**Could a rectangular matrix have a unique solution?**

<details><summary>Show explanation</summary>

Yes, for a consistent target if it has full column rank (r = n), requiring m ≥ n. This particular 3 × 4 matrix has r = 2, so every consistent target has infinitely many solutions.

</details>